# MoMo Fraud Detection — Hyperparameter Tuning
**Step 6 of 6**

## 0. Install Dependencies

In [ ]:
import subprocess,sys
for pkg in ['pandas','numpy','scikit-learn','xgboost','matplotlib']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
print('Ready')

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
SEED=42

## 2. Load Data

In [ ]:
df = pd.read_csv('Pay-sim_features.csv')
X = df.drop(columns=['isFraud'])
y = df['isFraud']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=SEED,stratify=y)
smote=SMOTE(random_state=SEED)
X_train_sm,y_train_sm=smote.fit_resample(X_train,y_train)
print('Data ready')

## 3. Define Hyperparameter Search Space

In [ ]:
param_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [3, 5, 7],
    'learning_rate':    [0.01, 0.05, 0.1],
    'subsample':        [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'scale_pos_weight': [1, 5, 10]
}
print('Search space defined')

## 4. Run RandomizedSearchCV

In [ ]:
xgb = XGBClassifier(random_state=SEED, use_label_encoder=False, eval_metric='logloss')
search = RandomizedSearchCV(
    xgb, param_grid, n_iter=20,
    scoring='f1', cv=3,
    random_state=SEED, n_jobs=-1, verbose=1)
search.fit(X_train_sm, y_train_sm)
print('Best params:', search.best_params_)
print('Best CV F1 :', search.best_score_.round(4))

## 5. Evaluate Best Model

In [ ]:
best = search.best_estimator_
y_pred = best.predict(X_test)
y_prob = best.predict_proba(X_test)[:,1]
print(classification_report(y_test,y_pred,target_names=['Non-Fraud','Fraud']))
print('ROC-AUC:',roc_auc_score(y_test,y_prob).round(4))

## 6. Feature Importance

In [ ]:
feat_imp = pd.Series(best.feature_importances_, index=X.columns)
top20 = feat_imp.nlargest(20)
top20.sort_values().plot(kind='barh', figsize=(9,7), color='steelblue',
                         title='Top 20 Feature Importances (XGBoost)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 7. Save Final Tuned Model

In [ ]:
with open('models/xgboost_tuned.pkl','wb') as f:
    pickle.dump(best, f)
print('Tuned model saved -> models/xgboost_tuned.pkl')